# Projet Complet : Classification de la Tendance du S&P 500 (ETF SPY)## ContexteL'**ETF SPY

** (SPDR S&P 500) est le fonds indiciel le plus célèbre au monde. Il réplique la performance des 500 plus grandes entreprises américaines cotées en bourse. C'est un baromètre de l'économie américaine.## ObjectifPrédire la **tendance du SPY** le lendemain en 3 classes :- **Classe 0** : Baisse (variation < -0.3%)- 
**Classe 1** : Stable (variation entre -0.3% et +0.3%)- 
**Classe 2** : Hausse (variation > +0.3%)## Type de problème**Classification multi-classes à 3 catégories**## Plan du notebook1. **Phase 1** : Chargement et exploration des données2. 
**Phase 2** : Visualisation des données3. 
**Phase 3** : Création de la variable target (3 classes)4. 
**Phase 4** : Feature Engineering (indicateurs financiers)5.
**Phase 5** : Prétraitement des données (normalisation, split)6. 
**Phase 6** : Modélisation (5 modèles de classification)7. 
**Phase 7** : Comparaison et évaluation des modèles8. 
**Phase 8** : Application Streamlit (interface web interactive)

---
# Phase 1 : Chargement et Exploration des Données
---

## 1.1 Importation des bibliothèques

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

## 1.2 Chargement du dataset

Le fichier `spy_us.txt` est au format CSV (séparateur virgule) malgré son extension `.txt`.

In [ ]:
df = pd.read_csv("spy_us.txt")
df.head()

### Description des colonnes

| Colonne | Description |
|---|---|
| Date | Date de la cotation |
| Open | Prix d'ouverture de la journée |
| High | Prix le plus haut de la journée |
| Low | Prix le plus bas de la journée |
| Close | Prix de clôture de la journée |
| Volume | Nombre d'actions échangées |
| OpenInt | Open Interest (non utilisé pour les ETF) |

## 1.3 Exploration basique

In [ ]:
# Dimensions du dataset
print("Nombre de lignes :", df.shape[0])
print("Nombre de colonnes :", df.shape[1])

In [ ]:
# Types des colonnes
df.dtypes

In [ ]:
# Informations générales
df.info()

In [ ]:
# Convertir la colonne Date en format datetime
df['Date'] = pd.to_datetime(df['Date'])
print("Période couverte :")
print(f"  Début : {df['Date'].min()}")
print(f"  Fin   : {df['Date'].max()}")
print(f"  Durée : {(df['Date'].max() - df['Date'].min()).days} jours")

In [ ]:
# Afficher la colonne OpenInt (on va voir qu'elle est inutile)
print("Valeurs uniques dans OpenInt :", df['OpenInt'].unique())
print("=> Cette colonne ne contient que des 0, on va la supprimer")

In [ ]:
# Supprimer la colonne OpenInt qui n'apporte rien
df = df.drop(columns=['OpenInt'])
print("Colonnes restantes :", list(df.columns))

## 1.4 Statistiques descriptives

In [ ]:
df.describe()

**Interprétation** :
- Le prix a évolué entre environ 65$ (min) et 260$+ (max) sur 13 ans
- Le volume quotidien varie énormément (de quelques millions à plusieurs centaines de millions)
- Les colonnes Open, High, Low, Close sont très proches (cohérent pour un ETF stable)

## 1.5 Vérification des valeurs manquantes

In [ ]:
missing = df.isnull().sum()
print("Valeurs manquantes par colonne :")
print(missing)
print(f"\nTotal de valeurs manquantes : {missing.sum()}")

Aucune valeur manquante — le dataset est propre.

---
# Phase 2 : Visualisation des Données
---

## 2.1 Évolution du prix de clôture

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(df['Date'], df['Close'], color='steelblue', linewidth=1)
plt.title("Évolution du prix de clôture de l'ETF SPY (2005-2017)", fontsize=14)
plt.xlabel("Date")
plt.ylabel("Prix ($)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Observation** : On voit clairement la crise de 2008 (chute vers 70$) puis la reprise progressive jusqu'à plus de 250$ en 2017.

## 2.2 Évolution du volume

In [ ]:
plt.figure(figsize=(14, 5))
plt.fill_between(df['Date'], df['Volume']/1e6, color='coral', alpha=0.6)
plt.title("Volume quotidien d'échanges (en millions d'actions)", fontsize=14)
plt.xlabel("Date")
plt.ylabel("Volume (millions)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Observation** : Les pics de volume correspondent aux périodes de crise (2008-2009) où les investisseurs paniquent et vendent massivement.

## 2.3 Distribution des prix

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

cols = ['Open', 'High', 'Low', 'Close']
for ax, col in zip(axes.flatten(), cols):
    ax.hist(df[col], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
    ax.set_title(f'Distribution de {col}')
    ax.set_xlabel('Prix ($)')
    ax.set_ylabel('Fréquence')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2.4 Boxplot pour détecter les outliers

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 5))
cols_all = ['Open', 'High', 'Low', 'Close', 'Volume']

for ax, col in zip(axes, cols_all):
    sns.boxplot(y=df[col], ax=ax, color='lightblue')
    ax.set_title(col)

plt.tight_layout()
plt.show()

**Observation** : La variable `Volume` a beaucoup d'outliers (jours de crise avec des volumes énormes). Les prix n'ont pas d'outliers extrêmes car le SPY est un ETF très stable.

## 2.5 Relations entre les variables (matrice de corrélation)

In [ ]:
plt.figure(figsize=(10, 8))
corr = df.drop(columns=['Date']).corr()
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', linewidths=0.5, 
            vmin=-1, vmax=1, center=0)
plt.title('Matrice de corrélation entre les variables')
plt.tight_layout()
plt.show()

**Observation** : 
- Open, High, Low, Close sont **quasi parfaitement corrélés** (coef ≈ 1). C'est logique car ce sont 4 prix pris à des moments différents de la même journée.
- Le Volume est **négativement corrélé** aux prix (quand les prix baissent, le volume augmente = panique).

---
# Phase 3 : Création de la Variable Target
---

## 3.1 Logique de construction

On calcule la **variation du prix de clôture** entre aujourd'hui et demain :

$$\text{Variation}_t = \frac{\text{Close}_{t+1} - \text{Close}_t}{\text{Close}_t}$$

Puis on classifie en 3 catégories :

| Variation | Classe | Label |
|---|---|---|
| > +0.3% | 2 | Hausse |
| entre -0.3% et +0.3% | 1 | Stable |
| < -0.3% | 0 | Baisse |

In [ ]:
# Trier par date (important pour les calculs temporels)
df = df.sort_values('Date').reset_index(drop=True)

# Calculer le prix de clôture DEMAIN (shift -1 = ligne suivante)
df['Close_tomorrow'] = df['Close'].shift(-1)

# Calculer la variation en pourcentage
df['Daily_return'] = (df['Close_tomorrow'] - df['Close']) / df['Close'] * 100

# Aperçu
df[['Date', 'Close', 'Close_tomorrow', 'Daily_return']].head(10)

## 3.2 Distribution de la variation journalière

In [ ]:
# Statistiques
print("Statistiques de la variation journalière (%) :")
print(df['Daily_return'].describe())

In [ ]:
# Histogramme de la distribution des variations
plt.figure(figsize=(12, 6))
plt.hist(df['Daily_return'].dropna(), bins=100, color='steelblue', 
         edgecolor='black', alpha=0.7)

# Ajouter les lignes des seuils
plt.axvline(x=-0.3, color='red', linestyle='--', linewidth=2, label='Seuil Baisse (-0.3%)')
plt.axvline(x=0.3, color='green', linestyle='--', linewidth=2, label='Seuil Hausse (+0.3%)')
plt.axvline(x=0, color='black', linestyle='-', linewidth=1, alpha=0.5)

plt.title('Distribution des variations journalières du SPY', fontsize=14)
plt.xlabel('Variation (%)')
plt.ylabel('Fréquence')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3.3 Création de la target à 3 classes

In [ ]:
def categorize_trend(variation):
    if pd.isna(variation):
        return np.nan
    elif variation > 0.3:
        return 2   # Hausse
    elif variation < -0.3:
        return 0   # Baisse
    else:
        return 1   # Stable

df['Target'] = df['Daily_return'].apply(categorize_trend)

# Vérifier le résultat
df[['Date', 'Close', 'Daily_return', 'Target']].head(10)

In [ ]:
# Répartition des 3 classes
print("Répartition des classes (nombre) :")
print(df['Target'].value_counts().sort_index())

print("\nRépartition des classes (pourcentage) :")
print((df['Target'].value_counts(normalize=True).sort_index() * 100).round(2))

In [ ]:
# Visualiser la répartition
class_labels = {0: 'Baisse', 1: 'Stable', 2: 'Hausse'}
colors = ['#E24B4A', '#888780', '#1D9E75']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Bar plot
counts = df['Target'].value_counts().sort_index()
axes[0].bar([class_labels[i] for i in counts.index], counts.values, color=colors)
axes[0].set_title('Nombre d\'observations par classe')
axes[0].set_ylabel('Nombre')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=[class_labels[i] for i in counts.index], 
            colors=colors, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Proportion des classes')

plt.tight_layout()
plt.show()

**Interprétation** : Le dataset est **bien équilibré** entre les 3 classes (environ 28% / 36% / 36%). C'est excellent pour un problème de classification — pas besoin de techniques de rééquilibrage comme SMOTE.

---
# Phase 4 : Feature Engineering
---

## 4.1 Pourquoi créer de nouvelles features ?

Les colonnes brutes (Open, High, Low, Close, Volume) ne suffisent pas pour bien prédire. Les traders utilisent des **indicateurs techniques** calculés à partir de ces prix. On va en créer les principaux.

## 4.2 Features de base (différences et ratios)

In [ ]:
# Range journalier (High - Low) : volatilité intra-jour
df['Daily_range'] = df['High'] - df['Low']

# Spread Close-Open : tendance de la journée
df['Close_open_spread'] = df['Close'] - df['Open']

# Variation haute-basse en pourcentage
df['Range_pct'] = (df['High'] - df['Low']) / df['Low'] * 100

# Aperçu
df[['Date', 'Open', 'High', 'Low', 'Close', 'Daily_range', 
    'Close_open_spread', 'Range_pct']].head()

## 4.3 Moyennes mobiles (Moving Averages)

Les moyennes mobiles lissent les variations pour détecter les tendances.

In [ ]:
# Moyenne mobile sur 5 jours (court terme)
df['MA_5'] = df['Close'].rolling(window=5).mean()

# Moyenne mobile sur 20 jours (moyen terme)
df['MA_20'] = df['Close'].rolling(window=20).mean()

# Moyenne mobile sur 50 jours (long terme)
df['MA_50'] = df['Close'].rolling(window=50).mean()

# Visualiser les moyennes mobiles sur une période
plt.figure(figsize=(14, 6))
subset = df[df['Date'] >= '2015-01-01']
plt.plot(subset['Date'], subset['Close'], label='Prix de clôture', alpha=0.6, color='gray')
plt.plot(subset['Date'], subset['MA_5'], label='MA 5 jours', color='blue')
plt.plot(subset['Date'], subset['MA_20'], label='MA 20 jours', color='orange')
plt.plot(subset['Date'], subset['MA_50'], label='MA 50 jours', color='red')
plt.title('Prix et moyennes mobiles (2015-2017)')
plt.xlabel('Date')
plt.ylabel('Prix ($)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4.4 Volatilité

In [ ]:
# Écart-type des rendements sur 20 jours = volatilité
df['Volatility_20'] = df['Daily_return'].rolling(window=20).std()

plt.figure(figsize=(14, 5))
plt.plot(df['Date'], df['Volatility_20'], color='crimson', linewidth=1)
plt.title('Volatilité (écart-type des rendements sur 20 jours)')
plt.xlabel('Date')
plt.ylabel('Volatilité (%)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Observation** : La volatilité explose pendant la crise de 2008-2009. Cet indicateur est très utile pour détecter les périodes de stress du marché.

## 4.5 Momentum (variations passées)

In [ ]:
# Variation sur 1 jour (hier vs aujourd'hui)
df['Return_1d'] = df['Close'].pct_change(periods=1) * 100

# Variation sur 5 jours
df['Return_5d'] = df['Close'].pct_change(periods=5) * 100

# Variation sur 20 jours
df['Return_20d'] = df['Close'].pct_change(periods=20) * 100

df[['Date', 'Close', 'Return_1d', 'Return_5d', 'Return_20d']].tail(10)

## 4.6 Volume relatif

In [ ]:
# Ratio volume / volume moyen sur 20 jours
df['Volume_ratio'] = df['Volume'] / df['Volume'].rolling(window=20).mean()

# Un ratio > 1 signifie que le volume d'aujourd'hui est au-dessus de la moyenne
print("Statistiques du ratio de volume :")
print(df['Volume_ratio'].describe())

## 4.7 Résumé des features créées

In [ ]:
features_created = [
    'Daily_range', 'Close_open_spread', 'Range_pct',
    'MA_5', 'MA_20', 'MA_50',
    'Volatility_20',
    'Return_1d', 'Return_5d', 'Return_20d',
    'Volume_ratio'
]

print(f"Nombre de features créées : {len(features_created)}")
for f in features_created:
    print(f"  - {f}")

---
# Phase 5 : Prétraitement des Données
---

## 5.1 Gestion des valeurs manquantes

Nos calculs de moyennes mobiles et de rendements créent des NaN au début du dataset (les premiers jours ne peuvent pas avoir de MA_50 par exemple).

In [ ]:
# Compter les NaN par colonne
print("Valeurs manquantes par colonne :")
print(df.isnull().sum()[df.isnull().sum() > 0].sort_values(ascending=False))

In [ ]:
# Dimensions avant nettoyage
print("Avant nettoyage :", df.shape)

# Supprimer les lignes avec des NaN (début et fin du dataset)
df_clean = df.dropna().reset_index(drop=True)

print("Après nettoyage :", df_clean.shape)
print(f"Lignes perdues : {df.shape[0] - df_clean.shape[0]}")

## 5.2 Sélection des features et de la target

In [ ]:
# Liste des features qu'on va utiliser pour la classification
feature_cols = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'Daily_range', 'Close_open_spread', 'Range_pct',
    'MA_5', 'MA_20', 'MA_50',
    'Volatility_20',
    'Return_1d', 'Return_5d', 'Return_20d',
    'Volume_ratio'
]

X = df_clean[feature_cols]
y = df_clean['Target'].astype(int)

print("X shape :", X.shape)
print("y shape :", y.shape)
print("\nFeatures utilisées :", feature_cols)

## 5.3 Split Train / Test

Pour les **séries temporelles**, on ne split PAS aléatoirement ! On garde l'ordre chronologique :
- **Train** = premières 80% des données (2005 à ~2015)
- **Test** = dernières 20% des données (~2015 à 2017)

Cela simule la réalité : on entraîne sur le passé, on teste sur le futur.

In [ ]:
from sklearn.model_selection import train_test_split

# shuffle=False pour garder l'ordre chronologique
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

print("Train :", X_train.shape, "| Dates :", 
      df_clean['Date'].iloc[0].date(), "->", 
      df_clean['Date'].iloc[len(X_train)-1].date())
print("Test  :", X_test.shape, "| Dates :", 
      df_clean['Date'].iloc[len(X_train)].date(), "->",
      df_clean['Date'].iloc[-1].date())

In [ ]:
# Vérifier la répartition des classes dans chaque ensemble
print("Répartition dans TRAIN :")
print(y_train.value_counts(normalize=True).sort_index().round(3))

print("\nRépartition dans TEST :")
print(y_test.value_counts(normalize=True).sort_index().round(3))

## 5.4 Normalisation avec StandardScaler

$$X_{normalized} = \frac{X - \mu}{\sigma}$$

Après normalisation : moyenne = 0, écart-type = 1.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# fit_transform sur le train UNIQUEMENT
X_train_scaled = scaler.fit_transform(X_train)

# transform (pas fit !) sur le test
X_test_scaled = scaler.transform(X_test)

# Convertir en DataFrame pour lisibilité
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=feature_cols)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=feature_cols)

print("Données normalisées :")
print(X_train_scaled_df.describe().round(3).loc[['mean', 'std']])

**Important** : On fait `fit_transform` uniquement sur le TRAIN, puis `transform` sur le TEST. Sinon on "triche" en utilisant les statistiques du test pour entraîner.

## 5.5 Visualisation avant/après normalisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Avant normalisation
X_train[['Close', 'Volume', 'Return_1d']].boxplot(ax=axes[0])
axes[0].set_title('Avant normalisation')
axes[0].set_ylabel('Valeur')
axes[0].set_yscale('log')

# Après normalisation
X_train_scaled_df[['Close', 'Volume', 'Return_1d']].boxplot(ax=axes[1])
axes[1].set_title('Après normalisation')
axes[1].set_ylabel('Valeur normalisée')

plt.tight_layout()
plt.show()

## 5.6 Récapitulatif du prétraitement

In [ ]:
print("=" * 60)
print("RÉCAPITULATIF DU DATASET PRÊT POUR L'ENTRAÎNEMENT")
print("=" * 60)
print(f"Features utilisées      : {len(feature_cols)}")
print(f"Classes (target)        : 3 (Baisse, Stable, Hausse)")
print(f"Taille totale           : {len(X)} observations")
print(f"Ensemble d'entraînement : {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Ensemble de test        : {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")
print(f"Normalisation           : StandardScaler (moyenne=0, std=1)")
print("=" * 60)

---
# Phase 6 : Modélisation — Entraînement de 5 Modèles de Classification
---

## 6.1 Importation des outils de classification

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

## 6.2 Comprendre les métriques d'évaluation

Pour la **classification multi-classes** (3 classes ici), les métriques sont :

- **Accuracy** = Bonnes prédictions / Total
- **Precision** (classe k) : Quand on prédit la classe k, à quel point on a raison ?
- **Recall** (classe k) : Parmi les vrais exemples de la classe k, combien on en retrouve ?
- **F1-Score** (classe k) : Moyenne harmonique entre Precision et Recall

### Moyennes
- **Macro average** : moyenne simple des scores de chaque classe
- **Weighted average** : moyenne pondérée par la fréquence de chaque classe

---
## 6.3 Modèle 1 : Régression Logistique

La **Régression Logistique** est le modèle le plus simple de classification. Pour 3 classes, elle utilise la fonction **softmax** qui donne une probabilité pour chaque classe.

In [ ]:
model_logreg = LogisticRegression(max_iter=1000, random_state=42)
model_logreg.fit(X_train_scaled, y_train)

y_pred_logreg = model_logreg.predict(X_test_scaled)

print("=" * 50)
print("RÉGRESSION LOGISTIQUE")
print("=" * 50)
print(f"Accuracy  : {accuracy_score(y_test, y_pred_logreg):.4f}")
print(f"Precision (macro) : {precision_score(y_test, y_pred_logreg, average='macro'):.4f}")
print(f"Recall    (macro) : {recall_score(y_test, y_pred_logreg, average='macro'):.4f}")
print(f"F1-Score  (macro) : {f1_score(y_test, y_pred_logreg, average='macro'):.4f}")

In [ ]:
# Rapport détaillé par classe
print("Rapport de classification détaillé :\n")
print(classification_report(y_test, y_pred_logreg, 
                             target_names=['Baisse', 'Stable', 'Hausse']))

In [ ]:
# Matrice de confusion
cm_logreg = confusion_matrix(y_test, y_pred_logreg)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_logreg, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Baisse', 'Stable', 'Hausse'],
            yticklabels=['Baisse', 'Stable', 'Hausse'])
plt.title('Matrice de Confusion - Régression Logistique')
plt.xlabel('Prédiction')
plt.ylabel('Réalité')
plt.tight_layout()
plt.show()

### Comment lire la matrice de confusion 3x3 ?

Chaque case (i, j) contient le nombre d'observations qui ont pour vraie classe **i** et ont été prédites comme **j**.

**La diagonale** = bonnes prédictions. **Hors diagonale** = erreurs.

---
## 6.4 Modèle 2 : K-Nearest Neighbors (KNN)

Le **KNN** classe un jour en regardant les **K jours les plus similaires** dans l'historique.

In [ ]:
model_knn = KNeighborsClassifier(n_neighbors=5)
model_knn.fit(X_train_scaled, y_train)

y_pred_knn = model_knn.predict(X_test_scaled)

print("=" * 50)
print("K-NEAREST NEIGHBORS (K=5)")
print("=" * 50)
print(f"Accuracy  : {accuracy_score(y_test, y_pred_knn):.4f}")
print(f"F1-Score (macro) : {f1_score(y_test, y_pred_knn, average='macro'):.4f}")
print("\nRapport détaillé :")
print(classification_report(y_test, y_pred_knn, 
                             target_names=['Baisse', 'Stable', 'Hausse']))

### Recherche du meilleur K

In [ ]:
# Tester plusieurs valeurs de K
k_values = range(1, 31)
accuracies = []
f1_scores = []

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)
    accuracies.append(accuracy_score(y_test, pred))
    f1_scores.append(f1_score(y_test, pred, average='macro'))

plt.figure(figsize=(12, 6))
plt.plot(k_values, accuracies, marker='o', label='Accuracy', color='steelblue')
plt.plot(k_values, f1_scores, marker='s', label='F1-Score (macro)', color='coral')
plt.xlabel('Nombre de voisins (K)')
plt.ylabel('Score')
plt.title('Performance du KNN en fonction de K')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_k = k_values[np.argmax(accuracies)]
print(f"\nMeilleur K : {best_k} (Accuracy = {max(accuracies):.4f})")

In [ ]:
# Réentraîner avec le meilleur K
model_knn_best = KNeighborsClassifier(n_neighbors=best_k)
model_knn_best.fit(X_train_scaled, y_train)
y_pred_knn_best = model_knn_best.predict(X_test_scaled)

# Matrice de confusion
cm_knn = confusion_matrix(y_test, y_pred_knn_best)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Baisse', 'Stable', 'Hausse'],
            yticklabels=['Baisse', 'Stable', 'Hausse'])
plt.title(f'Matrice de Confusion - KNN (K={best_k})')
plt.xlabel('Prédiction')
plt.ylabel('Réalité')
plt.tight_layout()
plt.show()

---
## 6.5 Modèle 3 : Arbre de Décision

L'**arbre de décision** pose une série de questions (Si Volume > X, alors...) pour classer les données. Très **interprétable**.

In [ ]:
model_tree = DecisionTreeClassifier(max_depth=5, random_state=42)
model_tree.fit(X_train_scaled, y_train)

y_pred_tree = model_tree.predict(X_test_scaled)

print("=" * 50)
print("ARBRE DE DÉCISION (max_depth=5)")
print("=" * 50)
print(f"Accuracy  : {accuracy_score(y_test, y_pred_tree):.4f}")
print(f"F1-Score (macro) : {f1_score(y_test, y_pred_tree, average='macro'):.4f}")
print("\nRapport détaillé :")
print(classification_report(y_test, y_pred_tree, 
                             target_names=['Baisse', 'Stable', 'Hausse']))

In [ ]:
# Importance des features
importance_tree = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': model_tree.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(12, 7))
plt.barh(importance_tree['Feature'], importance_tree['Importance'], color='teal')
plt.xlabel('Importance')
plt.title('Importance des features selon l\'arbre de décision')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 5 features les plus importantes :")
print(importance_tree.head())

---
## 6.6 Modèle 4 : Random Forest

Un **Random Forest** = plusieurs arbres de décision combinés. Chaque arbre vote, et la classe majoritaire gagne.

In [ ]:
model_rf = RandomForestClassifier(n_estimators=100, max_depth=10, 
                                    random_state=42, n_jobs=-1)
model_rf.fit(X_train_scaled, y_train)

y_pred_rf = model_rf.predict(X_test_scaled)

print("=" * 50)
print("RANDOM FOREST (100 arbres, max_depth=10)")
print("=" * 50)
print(f"Accuracy  : {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"F1-Score (macro) : {f1_score(y_test, y_pred_rf, average='macro'):.4f}")
print("\nRapport détaillé :")
print(classification_report(y_test, y_pred_rf, 
                             target_names=['Baisse', 'Stable', 'Hausse']))

In [ ]:
# Importance des features selon le Random Forest
importance_rf = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': model_rf.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(12, 7))
plt.barh(importance_rf['Feature'], importance_rf['Importance'], color='purple')
plt.xlabel('Importance')
plt.title('Importance des features selon le Random Forest')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

---
## 6.7 Modèle 5 : SVM (Support Vector Machine)

Le **SVM** cherche la meilleure "frontière" (hyperplan) qui sépare les classes dans l'espace des features.

In [ ]:
model_svm = SVC(kernel='rbf', C=1.0, random_state=42)
model_svm.fit(X_train_scaled, y_train)

y_pred_svm = model_svm.predict(X_test_scaled)

print("=" * 50)
print("SVM (kernel='rbf')")
print("=" * 50)
print(f"Accuracy  : {accuracy_score(y_test, y_pred_svm):.4f}")
print(f"F1-Score (macro) : {f1_score(y_test, y_pred_svm, average='macro'):.4f}")
print("\nRapport détaillé :")
print(classification_report(y_test, y_pred_svm, 
                             target_names=['Baisse', 'Stable', 'Hausse']))

---
# Phase 7 : Comparaison Globale des Modèles
---

In [ ]:
# Rassembler toutes les métriques dans un tableau
models_results = {
    'Régression Logistique': y_pred_logreg,
    f'KNN (K={best_k})': y_pred_knn_best,
    'Arbre de Décision': y_pred_tree,
    'Random Forest': y_pred_rf,
    'SVM': y_pred_svm
}

results_df = pd.DataFrame({
    'Modèle': list(models_results.keys()),
    'Accuracy': [accuracy_score(y_test, p) for p in models_results.values()],
    'Precision (macro)': [precision_score(y_test, p, average='macro') for p in models_results.values()],
    'Recall (macro)': [recall_score(y_test, p, average='macro') for p in models_results.values()],
    'F1-Score (macro)': [f1_score(y_test, p, average='macro') for p in models_results.values()],
    'F1-Score (weighted)': [f1_score(y_test, p, average='weighted') for p in models_results.values()]
})

results_df = results_df.sort_values('F1-Score (macro)', ascending=False).reset_index(drop=True)
print("Classement des modèles par F1-Score macro :\n")
print(results_df.round(4).to_string(index=False))

In [ ]:
# Visualiser la comparaison
fig, ax = plt.subplots(figsize=(14, 7))
metrics = ['Accuracy', 'Precision (macro)', 'Recall (macro)', 'F1-Score (macro)']
x = np.arange(len(results_df))
width = 0.2

for i, metric in enumerate(metrics):
    ax.bar(x + i*width, results_df[metric], width, label=metric)

ax.set_xlabel('Modèles')
ax.set_ylabel('Score')
ax.set_title('Comparaison des modèles de classification')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(results_df['Modèle'], rotation=15, ha='right')
ax.legend(loc='lower right')
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Matrices de confusion de tous les modèles côte à côte
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

for ax, (name, pred) in zip(axes, models_results.items()):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Baisse', 'Stable', 'Hausse'],
                yticklabels=['Baisse', 'Stable', 'Hausse'])
    acc = accuracy_score(y_test, pred)
    ax.set_title(f'{name}\n(Accuracy = {acc:.3f})')
    ax.set_xlabel('Prédiction')
    ax.set_ylabel('Réalité')

# Masquer la dernière case inutile
axes[-1].set_visible(False)

plt.tight_layout()
plt.show()

## Analyse du meilleur modèle

In [ ]:
# Identifier le meilleur modèle
best_model_name = results_df.iloc[0]['Modèle']
best_pred = models_results[best_model_name]

print(f"Meilleur modèle : {best_model_name}")
print(f"F1-Score macro : {results_df.iloc[0]['F1-Score (macro)']:.4f}")
print(f"Accuracy       : {results_df.iloc[0]['Accuracy']:.4f}")

# Taux de bonnes prédictions par classe
print("\nAnalyse par classe :")
for cls, label in zip([0, 1, 2], ['Baisse', 'Stable', 'Hausse']):
    mask = y_test == cls
    correct = (best_pred[mask] == cls).sum()
    total = mask.sum()
    print(f"  {label} : {correct}/{total} bien prédites ({correct/total*100:.1f}%)")

---# Conclusion---## Ce qu'on a accompli1. **Chargé et exploré** le dataset SPY (13 ans de données boursières)2. **Visualisé** les prix, volumes, distributions et corrélations3. **Créé une target à 3 classes** équilibrée (Baisse / Stable / Hausse)4. **Généré 16 features** dont 11 indicateurs techniques (moyennes mobiles, volatilité, momentum)5. **Prétraité les données** : gestion des NaN, normalisation StandardScaler, split chronologique6. **Entraîné 5 modèles** de classification (Régression Logistique, KNN, Arbre de Décision, Random Forest, SVM)7. **Évalué et comparé** avec Accuracy, Precision, Recall, F1-Score, matrices de confusion8. **Créé une application Streamlit** interactive pour présenter le projet## Points importants- La **prédiction des marchés est très difficile** — obtenir une accuracy > 50% pour 3 classes est déjà bon- Un modèle **simple** (Régression Logistique) peut surpasser les modèles complexes quand il y a peu de signal- Le **split chronologique** est essentiel pour les séries temporelles- L'application **Streamlit** permet de présenter le projet de manière professionnelle et interactive---